# Pdf loader

In [16]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [21]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: NLP_Detailed_Overview.pdf
Loaded 1 pages

Processing: RAG_Detailed_Overview.pdf
Loaded 2 pages

Processing: Vector_Database_Detailed_Overview.pdf
Loaded 1 pages

Total documents loaded: 4


In [22]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-03-01T18:47:31+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-01T18:47:31+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\NLP_Detailed_Overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Detailed_Overview.pdf', 'file_type': 'pdf'}, page_content='Natural Language Processing (NLP)\nFoundations of Natural Language Processing (NLP)\nNatural Language Processing (NLP) is a field of artificial intelligence focused on enabling machines\nto understand, interpret, and generate human language. It combines linguistics, computer science,\nand machine learning.\nEarly NLP relied on rule-based systems, but modern NLP is dominated by deep learning\napproaches, especially transformer-based architectures.\nModern Architectures and Techniques\nTransformers revolutionize

In [26]:
## Text splitting get into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    ## split documents into smaller chunks for better RAG performance

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ",""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content : {split_docs[0].page_content[:200]}....")
        print(f"Metadata : {split_docs[0].metadata}")

    return split_docs

In [27]:
chunks = split_documents(all_pdf_documents)
chunks

split 4 documents into 7 chunks

Example chunk:
Content : Natural Language Processing (NLP)
Foundations of Natural Language Processing (NLP)
Natural Language Processing (NLP) is a field of artificial intelligence focused on enabling machines
to understand, i....
Metadata : {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-03-01T18:47:31+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-01T18:47:31+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\NLP_Detailed_Overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Detailed_Overview.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-03-01T18:47:31+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-01T18:47:31+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\NLP_Detailed_Overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Detailed_Overview.pdf', 'file_type': 'pdf'}, page_content='Natural Language Processing (NLP)\nFoundations of Natural Language Processing (NLP)\nNatural Language Processing (NLP) is a field of artificial intelligence focused on enabling machines\nto understand, interpret, and generate human language. It combines linguistics, computer science,\nand machine learning.\nEarly NLP relied on rule-based systems, but modern NLP is dominated by deep learning\napproaches, especially transformer-based architectures.\nModern Architectures and Techniques\nTransformers revolutionize

## Embedding and VectorStore DB

In [28]:
import numpy as np 
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [29]:
class EmbeddingManager:
    ''' Handles document embedding generation using SentenceTransformers'''
    def __init__(self, model_name : str = 'all-MiniLM-L6-v2'):


        """
    Initialize the embedding manager

    args: 
        model_name : HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()


    def _load_model(self):

        """Load the SentenceTransformer model"""

        try:
            print(f'Loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension : {self.model.get_sentence_embedding_dimension()}")
        
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, text: List[str]) -> np.ndarray:
        """ 
        generate embeddings for a list of texts

        Args : 
            texts : List of text strings to embed 

        Returns :
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings


## initialize embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension : 384


## VECTOR STORE

In [44]:
class VectorStore:
    # Manages document embeddings in a ChromaDB vector store

    def __init__(self, collection_name: str = "pdf_documents", persist_directory : str = "../DATA/vector_store"):

        """
        Initialize the vector store

        Args : 
            collection_name : Name of the ChromaDB collection

            persist_director : Directory to persist the vector store

        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        # initialize ChromaDB client and collection

        try:
            # Create persistent chromaDB client

            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description":"PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection : {self.collection_name}")

            print(f"Existing documents in collection : {self.collection.count()}")

        except Exception as e:
            print(f"error intializing vector store : {e}")
            raise

    
    def add_documents(self, documents: List[Any], embeddings : np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args : 
            Documents : List of Langchain documents
            embeddings : Corresponding embeddings for the documents

        """

        if len(documents) != len(embeddings):
            raise ValueError("no of documents must match no of embeddings")
    
        print(f"Adding {len(documents)} documents to vector store...")

        ## Prepare data for ChromaDB

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ### Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content

            documents_text.append(doc.page_content)

            # Embeddings 

            embeddings_list.append(embedding.tolist())

        ## Add to collection
        try: 
            self.collection.add(
                ids =ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text

            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection : {self.collection.count()}")

        except Exception as e:
            print(f'error adding documents to vector store: {e}')

            raise


vectorstore = VectorStore()
vectorstore
            


Vector store initialized. Collection : pdf_documents
Existing documents in collection : 0


In [45]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-03-01T18:47:31+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-01T18:47:31+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\NLP_Detailed_Overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'NLP_Detailed_Overview.pdf', 'file_type': 'pdf'}, page_content='Natural Language Processing (NLP)\nFoundations of Natural Language Processing (NLP)\nNatural Language Processing (NLP) is a field of artificial intelligence focused on enabling machines\nto understand, interpret, and generate human language. It combines linguistics, computer science,\nand machine learning.\nEarly NLP relied on rule-based systems, but modern NLP is dominated by deep learning\napproaches, especially transformer-based architectures.\nModern Architectures and Techniques\nTransformers revolutionize

In [ ]:
## Convert the text to embeddings

texts = [doc.page_content for doc in chunks]


## Generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 7 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embedding with shape: (7, 384)
Adding 7 documents to vector store...
successfully added 7 documents to vector store
total documents in collection : 7


## Retriever Pipeline from VectorStore 

In [ ]:
class RAGRetriever:
    # Handles query-based retrieval from the vector store

    def __init__(self, vector_store: vectorstore, embedding_manager : EmbeddingManager):
        """
        Initialize the retriever

        args :
            vector_store : Vector store containing document embeddings

            embedding_manager : Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

        